# Auto ML Pipeline
Load a CSV, inspect the data, detect the target column, decide classification or regression, preprocess the data, compare several ML models including ANN and tree-based methods, run clustering, and show the final metrics and plots.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from IPython.display import display
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import AdaBoostClassifier, AdaBoostRegressor, ExtraTreesClassifier, ExtraTreesRegressor, GradientBoostingClassifier, GradientBoostingRegressor, HistGradientBoostingClassifier, HistGradientBoostingRegressor, RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, mean_absolute_error, mean_squared_error, r2_score, f1_score, silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

try:
    from xgboost import XGBClassifier, XGBRegressor
    HAS_XGBOOST = True
except Exception:
    HAS_XGBOOST = False

try:
    from lightgbm import LGBMClassifier, LGBMRegressor
    HAS_LIGHTGBM = True
except Exception:
    HAS_LIGHTGBM = False

pd.set_option("display.max_columns", None)
sns.set_style("whitegrid")
np.random.seed(42)
tf.random.set_seed(42)


## 1) Load Data
Set `DATA_SOURCE` to a CSV link or local CSV path. You can also set `TARGET_COL` if you already know the target column.

In [ ]:
DATA_SOURCE = "mall_customers.csv"
TARGET_COL = "Gender"  # Example target

if not DATA_SOURCE or DATA_SOURCE == "paste_csv_url_or_path_here":
    raise ValueError("Set DATA_SOURCE to a CSV URL or local CSV path.")

df = pd.read_csv(DATA_SOURCE)
print("Shape:", df.shape)
display(df.head())
display(df.sample(min(len(df), 5), random_state=42))

summary = df.describe(include="all").T
summary["missing"] = df.isna().sum()
display(summary)

# Feature Distribution Plots
num_vars = df.select_dtypes(include="number").columns
if len(num_vars) > 0:
    rows = (len(num_vars) + 2) // 3
    plt.figure(figsize=(15, 4 * rows))
    for i, col in enumerate(num_vars):
        plt.subplot(rows, 3, i + 1)
        sns.histplot(df[col], kde=True, color="skyblue")
        plt.title(f"Distribution of {col}")
    plt.tight_layout()
    plt.show()

missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 0]
if not missing_pct.empty:
    plt.figure(figsize=(10, max(3, 0.35 * len(missing_pct))))
    missing_pct.sort_values().plot(kind="barh", color="#4C78A8")
    plt.title("Missing Value Percentage by Column")
    plt.xlabel("Percent missing")
    plt.tight_layout()
    plt.show()

numeric_preview = df.select_dtypes(include="number")
if not numeric_preview.empty:
    plt.figure(figsize=(10, 6))
    sns.heatmap(numeric_preview.corr(numeric_only=True), cmap="coolwarm", center=0)
    plt.title("Correlation Heatmap")
    plt.tight_layout()
    plt.show()


## 2) Detect Target and Task
The notebook checks common target names first and then falls back to the last column. Small-cardinality objects are treated as classification; everything else is treated as regression.

In [ ]:
def detect_target_column(frame, target_col=None):
    if target_col and target_col in frame.columns:
        return target_col
    common = ["target", "label", "class", "y", "outcome", "response"]
    lowered = {c.lower(): c for c in frame.columns}
    for name in common:
        if name in lowered:
            return lowered[name]
    return frame.columns[-1]


def detect_task(y):
    if y.dtype == "object" or str(y.dtype).startswith("category") or y.dtype == "bool":
        return "classification"
    unique_count = y.nunique(dropna=True)
    if unique_count <= 20 and unique_count / max(len(y), 1) < 0.05:
        return "classification"
    return "regression"


def infer_classification_type(y):
    unique_count = len(np.unique(y))
    return "binary" if unique_count == 2 else "multiclass"


target_col = detect_target_column(df, TARGET_COL)
df = df.dropna(subset=[target_col]).copy()
y_raw = df[target_col]
task = detect_task(y_raw)
X = df.drop(columns=[target_col])
class_type = infer_classification_type(LabelEncoder().fit_transform(y_raw.astype(str))) if task == "classification" else None

print("Target column:", target_col)
print("Task:", task)
if class_type:
    print("Classification type:", class_type)


## 3) Clean and Encode
Missing values are imputed, numeric fields are scaled, and categorical fields are one-hot encoded. The processed matrix is then shared across the model zoo and clustering steps.

In [ ]:
num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = X.select_dtypes(exclude="number").columns.tolist()

if task == "classification":
    y = LabelEncoder().fit_transform(y_raw.astype(str))
else:
    y = pd.to_numeric(y_raw, errors="coerce")
    keep = y.notna()
    X = X.loc[keep].copy()
    y = y.loc[keep].astype(float)

if task == "classification":
    class_type = "binary" if len(np.unique(y)) == 2 else "multiclass"

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols),
], remainder="drop")

split_args = dict(test_size=0.2, random_state=42)
if task == "classification":
    split_args["stratify"] = y
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, **split_args)

val_args = dict(test_size=0.2, random_state=42)
if task == "classification":
    val_args["stratify"] = y_train_full
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, **val_args)

X_train = preprocess.fit_transform(X_train)
X_val = preprocess.transform(X_val)
X_test = preprocess.transform(X_test)

if hasattr(X_train, "toarray"):
    X_train = X_train.toarray()
    X_val = X_val.toarray()
    X_test = X_test.toarray()

feature_names = preprocess.get_feature_names_out()
print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)
print("Feature count:", len(feature_names))


## 4) Train and Compare Models
A small model zoo is trained on the same preprocessed data so the notebook can compare decision trees, random forests, boosting methods, k-NN, XGBoost when available, and an ANN.

In [ ]:
def build_ann_model(input_dim, task, class_type, n_classes=2, units=64, layers_count=2, dropout=0.2, lr=1e-3):
    model = keras.Sequential([layers.Input(shape=(input_dim,))])
    current_units = units
    for _ in range(layers_count):
        model.add(layers.Dense(current_units, activation="relu"))
        model.add(layers.Dropout(dropout))
        current_units = max(current_units // 2, 8)
    if task == "classification":
        if class_type == "binary":
            model.add(layers.Dense(1, activation="sigmoid"))
            loss = "binary_crossentropy"
            metrics = [keras.metrics.BinaryAccuracy(name="accuracy")]
        else:
            model.add(layers.Dense(n_classes, activation="softmax"))
            loss = "sparse_categorical_crossentropy"
            metrics = ["accuracy"]
    else:
        model.add(layers.Dense(1, activation="linear"))
        loss = "mse"
        metrics = ["mae"]
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr), loss=loss, metrics=metrics)
    return model


def get_classification_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted"),
    }


def get_regression_metrics(y_true, y_pred):
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    return {
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": rmse,
        "r2": r2_score(y_true, y_pred),
    }


def make_model_factories(task, class_type, n_classes):
    if task == "classification":
        factories = [
            ("Decision Tree", lambda: DecisionTreeClassifier(random_state=42)),
            ("Random Forest", lambda: RandomForestClassifier(n_estimators=250, random_state=42, n_jobs=-1)),
            ("Extra Trees", lambda: ExtraTreesClassifier(n_estimators=250, random_state=42, n_jobs=-1)),
            ("Gradient Boosting", lambda: GradientBoostingClassifier(random_state=42)),
            ("AdaBoost", lambda: AdaBoostClassifier(random_state=42)),
            ("HistGradientBoosting", lambda: HistGradientBoostingClassifier(random_state=42)),
            ("kNN", lambda: KNeighborsClassifier(n_neighbors=7, weights="distance")),
        ]
        if HAS_XGBOOST:
            if class_type == "binary":
                factories.append((
                    "XGBoost",
                    lambda: XGBClassifier(
                        n_estimators=300,
                        max_depth=4,
                        learning_rate=0.05,
                        subsample=0.85,
                        colsample_bytree=0.85,
                        eval_metric="logloss",
                        random_state=42,
                        tree_method="hist",
                    ),
                ))
            else:
                factories.append((
                    "XGBoost",
                    lambda: XGBClassifier(
                        n_estimators=300,
                        max_depth=4,
                        learning_rate=0.05,
                        subsample=0.85,
                        colsample_bytree=0.85,
                        eval_metric="mlogloss",
                        objective="multi:softprob",
                        num_class=n_classes,
                        random_state=42,
                        tree_method="hist",
                    ),
                ))
        if HAS_LIGHTGBM:
            factories.append(("LightGBM", lambda: LGBMClassifier(n_estimators=300, random_state=42, verbosity=-1)))
    else:
        factories = [
            ("Decision Tree", lambda: DecisionTreeRegressor(random_state=42)),
            ("Random Forest", lambda: RandomForestRegressor(n_estimators=250, random_state=42, n_jobs=-1)),
            ("Extra Trees", lambda: ExtraTreesRegressor(n_estimators=250, random_state=42, n_jobs=-1)),
            ("Gradient Boosting", lambda: GradientBoostingRegressor(random_state=42)),
            ("AdaBoost", lambda: AdaBoostRegressor(random_state=42)),
            ("HistGradientBoosting", lambda: HistGradientBoostingRegressor(random_state=42)),
            ("kNN", lambda: KNeighborsRegressor(n_neighbors=7, weights="distance")),
        ]
        if HAS_XGBOOST:
            factories.append((
                "XGBoost",
                lambda: XGBRegressor(
                    n_estimators=300,
                    max_depth=4,
                    learning_rate=0.05,
                    subsample=0.85,
                    colsample_bytree=0.85,
                    random_state=42,
                    tree_method="hist",
                ),
            ))
    return factories


def evaluate_predictions(task, y_true, y_pred):
    if task == "classification":
        return get_classification_metrics(y_true, y_pred)
    return get_regression_metrics(y_true, y_pred)


n_classes = len(np.unique(y)) if task == "classification" else None
model_results = []
best_model = None
best_model_name = None
best_model_kind = None
best_history = None

selection_key = "f1_weighted" if task == "classification" else "rmse"
best_score = -np.inf if task == "classification" else np.inf

for name, factory in make_model_factories(task, class_type, n_classes or 0):
    model = factory()
    model.fit(X_train, y_train)
    val_pred = model.predict(X_val)
    metrics = evaluate_predictions(task, y_val, val_pred)
    score = metrics[selection_key]
    row = {"model": name, **metrics}
    row["selection_score"] = score
    model_results.append(row)
    if (task == "classification" and score > best_score) or (task == "regression" and score < best_score):
        best_score = score
        best_model = model
        best_model_name = name
        best_model_kind = "sklearn"
        best_history = None

ann_configs = [
    {"units": 64, "layers_count": 2, "dropout": 0.2, "lr": 1e-3},
    {"units": 128, "layers_count": 2, "dropout": 0.3, "lr": 5e-4},
    {"units": 128, "layers_count": 3, "dropout": 0.2, "lr": 1e-3},
]

for cfg in ann_configs:
    model = build_ann_model(X_train.shape[1], task, class_type, n_classes=n_classes or 2, **cfg)
    callbacks = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)]
    if task == "classification" and class_type == "binary":
        history = model.fit(
            X_train,
            y_train,
            validation_data=(X_val, y_val),
            epochs=40,
            batch_size=32,
            verbose=0,
            callbacks=callbacks,
        )
        val_pred = (model.predict(X_val, verbose=0).ravel() >= 0.5).astype(int)
    elif task == "classification":
        history = model.fit(
            X_train,
            y_train,
            validation_data=(X_val, y_val),
            epochs=40,
            batch_size=32,
            verbose=0,
            callbacks=callbacks,
        )
        val_pred = np.argmax(model.predict(X_val, verbose=0), axis=1)
    else:
        history = model.fit(
            X_train,
            y_train,
            validation_data=(X_val, y_val),
            epochs=40,
            batch_size=32,
            verbose=0,
            callbacks=callbacks,
        )
        val_pred = model.predict(X_val, verbose=0).ravel()

    metrics = evaluate_predictions(task, y_val, val_pred)
    score = metrics[selection_key]
    row = {"model": f"ANN {cfg['units']}x{cfg['layers_count']}", **metrics}
    row["selection_score"] = score
    model_results.append(row)
    if (task == "classification" and score > best_score) or (task == "regression" and score < best_score):
        best_score = score
        best_model = model
        best_model_name = row["model"]
        best_model_kind = "ann"
        best_history = history

results_df = pd.DataFrame(model_results)
if task == "classification":
    results_df = results_df.sort_values(["selection_score", "accuracy"], ascending=[False, False])
else:
    results_df = results_df.sort_values(["selection_score", "mae"], ascending=[True, True])

display(results_df)
print("Best model on validation:", best_model_name)

cluster_input = preprocess.transform(X)
if hasattr(cluster_input, "toarray"):
    cluster_input = cluster_input.toarray()

if cluster_input.shape[0] > 3000:
    sample_index = np.random.RandomState(42).choice(cluster_input.shape[0], 3000, replace=False)
    cluster_sample = cluster_input[sample_index]
else:
    cluster_sample = cluster_input

if cluster_sample.shape[1] > 1 and cluster_sample.shape[0] > 2:
    cluster_range = range(2, min(9, cluster_sample.shape[0]) + 1)
    cluster_scores = []
    cluster_models = []
    for k in cluster_range:
        kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
        labels = kmeans.fit_predict(cluster_sample)
        if len(np.unique(labels)) > 1:
            score = silhouette_score(cluster_sample, labels)
        else:
            score = np.nan
        cluster_scores.append(score)
        cluster_models.append(kmeans)

    # Hierarchical Clustering Comparison
    agglom = AgglomerativeClustering(n_clusters=3)
    agglom_labels = agglom.fit_predict(cluster_sample)
    if len(np.unique(agglom_labels)) > 1:
        agglom_score = silhouette_score(cluster_sample, agglom_labels)
        print(f"Agglomerative Clustering Silhouette Score (k=3): {agglom_score:.4f}")

    cluster_summary = pd.DataFrame({"k": list(cluster_range), "silhouette": cluster_scores})
    display(cluster_summary)

    best_cluster_row = cluster_summary.iloc[cluster_summary["silhouette"].idxmax()]
    best_k = int(best_cluster_row["k"])
    best_cluster_model = KMeans(n_clusters=best_k, n_init=10, random_state=42)
    cluster_labels = best_cluster_model.fit_predict(cluster_sample)

    plt.figure(figsize=(8, 4))
    sns.lineplot(data=cluster_summary, x="k", y="silhouette", marker="o")
    plt.title("KMeans Silhouette Score by Cluster Count")
    plt.tight_layout()
    plt.show()

    pca = PCA(n_components=2, random_state=42)
    cluster_2d = pca.fit_transform(cluster_sample)
    plt.figure(figsize=(8, 6))
    sns.scatterplot(x=cluster_2d[:, 0], y=cluster_2d[:, 1], hue=cluster_labels, palette="tab10", s=40, linewidth=0)
    plt.title(f"Cluster View with KMeans (k={best_k})")
    plt.xlabel("PCA 1")
    plt.ylabel("PCA 2")
    plt.legend(title="Cluster")
    plt.tight_layout()
    plt.show()
else:
    print("Clustering skipped because the transformed feature space is too small for a stable cluster view.")


## 5) Evaluate and Explain
The best model from the validation run is evaluated on the holdout test set, and the notebook shows the key diagnostic plots and feature importance when available.

In [ ]:
if best_model_kind == "ann":
    pred = best_model.predict(X_test, verbose=0)
else:
    pred = best_model.predict(X_test)

if task == "classification":
    if class_type == "binary":
        y_pred = (pred.ravel() >= 0.5).astype(int)
    else:
        y_pred = np.argmax(pred, axis=1) if pred.ndim > 1 else pred
    test_metrics = get_classification_metrics(y_test, y_pred)
    print("Best model:", best_model_name)
    print("Accuracy:", round(test_metrics["accuracy"], 4))
    print("Weighted F1:", round(test_metrics["f1_weighted"], 4))
    print(classification_report(y_test, y_pred))
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title(f"Confusion Matrix - {best_model_name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.show()
else:
    y_pred = pred.ravel()
    test_metrics = get_regression_metrics(y_test, y_pred)
    print("Best model:", best_model_name)
    print("MAE:", round(test_metrics["mae"], 4))
    print("RMSE:", round(test_metrics["rmse"], 4))
    print("R2:", round(test_metrics["r2"], 4))
    plt.figure(figsize=(6, 6))
    plt.scatter(y_test, y_pred, alpha=0.7)
    low = min(np.min(y_test), np.min(y_pred))
    high = max(np.max(y_test), np.max(y_pred))
    plt.plot([low, high], [low, high], "r--")
    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.title(f"Actual vs Predicted - {best_model_name}")
    plt.tight_layout()
    plt.show()

if best_model_kind == "ann" and best_history is not None:
    plt.figure(figsize=(8, 4))
    plt.plot(best_history.history["loss"], label="train_loss")
    plt.plot(best_history.history["val_loss"], label="val_loss")
    plt.legend()
    plt.title("Training Curve")
    plt.tight_layout()
    plt.show()

if hasattr(best_model, "feature_importances_"):
    importances = pd.Series(best_model.feature_importances_, index=feature_names).sort_values(ascending=False)
    top_importances = importances.head(15)
    plt.figure(figsize=(10, 6))
    sns.barplot(x=top_importances.values, y=top_importances.index, color="#4C78A8")
    plt.title(f"Top Feature Importances - {best_model_name}")
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()

print("Reasoning:")
if task == "classification":
    print("Higher accuracy and weighted F1 on the test split mean the selected model generalizes better to unseen classes.")
else:
    print("Lower MAE and RMSE with a higher R2 mean the selected model is capturing the target signal well.")
